# Smoking is bad

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data
DATA_PATH = Path.cwd() / "data" / "smoking_data.csv"
df = pd.read_csv(DATA_PATH)

In [ ]:
# TODO: what are the dimensions of the data frame?
df.shape

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string columns?
df.describe()

In [ ]:
# TODO: do not forget to check the null values
df.isna().sum()

In [ ]:
# TODO: prepare your data frame so that you can work with
df = df.drop(columns="Unnamed: 0")

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set instructions.
df.groupby("smoker")["outcome"].apply(lambda s: (s == "Alive").mean())

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, (smoker, label, color) in zip(
    axes, [("No", "Non-smokers", "tab:blue"), ("Yes", "Smokers", "tab:orange")]
):
    ax.hist(
        df.loc[df["smoker"] == smoker, "age"],
        bins=20,
        density=True,
        color=color,
        edgecolor="white",
    )
    ax.set_title(f"Age distribution of {label.lower()}")
    ax.set_xlabel("Age (years)")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Density")
plt.tight_layout()
plt.show()

## Why do smokers appear to survive more often?
The raw numbers say smokers are *more* likely to be alive than non-smokers, which cannot be a real health benefit. The most likely explanation is **confounding by age**: if smokers are younger on average, they have had less time to die. The plots below test this step by step.

In [ ]:
# Helper columns used by all the plots below
df["is_alive"] = (df["outcome"] == "Alive").astype(int)
df["is_smoker"] = (df["smoker"] == "Yes").astype(int)

age_bins = [17, 25, 35, 45, 55, 65, 75, 85]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76-84"]
df["age_band"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)

COLORS = {"No": "tab:blue", "Yes": "tab:orange"}
LABELS = {"No": "Non-smokers", "Yes": "Smokers"}

### 1. The puzzle: raw survival rate by smoking status

In [ ]:
raw = df.groupby("smoker")["is_alive"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    [LABELS[s] for s in raw.index],
    raw["mean"] * 100,
    color=[COLORS[s] for s in raw.index],
)
for bar, rate, n in zip(bars, raw["mean"], raw["count"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{rate:.1%}\n(n={n})",
        ha="center",
    )
ax.set_ylim(0, 100)
ax.set_ylabel("Proportion alive (%)")
ax.set_title("Raw survival rate: smokers vs. non-smokers")
ax.grid(axis="y", alpha=0.3)
plt.show()

### 2. Clue: smokers are younger on average (box plot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = ["No", "Yes"]
bp = ax.boxplot(
    [df.loc[df["smoker"] == s, "age"] for s in groups],
    tick_labels=[LABELS[s] for s in groups],
    patch_artist=True,
    showmeans=True,
)
for patch, s in zip(bp["boxes"], groups):
    patch.set_facecolor(COLORS[s])
    patch.set_alpha(0.6)
ax.set_ylabel("Age (years)")
ax.set_title("Age by smoking status (triangle = mean)")
ax.grid(axis="y", alpha=0.3)
plt.show()

### 3. Clue: non-smokers are over-represented in the oldest age bands (stacked bar)

In [ ]:
composition = pd.crosstab(df["smoker"], df["age_band"], normalize="index") * 100
composition.index = [LABELS[s] for s in composition.index]

ax = composition.plot.barh(
    stacked=True, colormap="viridis", figsize=(10, 3.5), edgecolor="white"
)
ax.set_xlabel("Share of group (%)")
ax.set_ylabel("")
ax.set_title("Age composition of each group")
ax.legend(title="Age band", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4. Age strongly predicts death, and smoking varies with age

In [ ]:
by_age = df.groupby("age_band", observed=True)[["is_alive", "is_smoker"]].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)

axes[0].bar(by_age.index, by_age["is_alive"], color="tab:green")
axes[0].set_title("Survival rate by age band")
axes[0].set_ylabel("Proportion alive (%)")

axes[1].bar(by_age.index, by_age["is_smoker"], color="tab:red")
axes[1].set_title("Smoking rate by age band")
axes[1].set_ylabel("Proportion who smoke (%)")

for ax in axes:
    ax.set_xlabel("Age band (years)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 5. Fair comparison: smokers vs. non-smokers *within* each age band (line plot)

In [ ]:
within = (
    df.groupby(["age_band", "smoker"], observed=True)["is_alive"].mean().unstack() * 100
)

fig, ax = plt.subplots(figsize=(9, 5))
for s in ["No", "Yes"]:
    ax.plot(within.index, within[s], marker="o", color=COLORS[s], label=LABELS[s])
ax.set_xlabel("Age band (years)")
ax.set_ylabel("Proportion alive (%)")
ax.set_title("Survival rate by age band and smoking status")
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### 6. The same comparison as a heatmap, with group sizes

In [ ]:
rate = df.pivot_table(
    index="smoker", columns="age_band", values="is_alive", aggfunc="mean", observed=True
)
size = df.pivot_table(
    index="smoker",
    columns="age_band",
    values="is_alive",
    aggfunc="count",
    observed=True,
)

fig, ax = plt.subplots(figsize=(11, 3.5))
im = ax.imshow(rate.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(rate.columns)), rate.columns)
ax.set_yticks(range(len(rate.index)), [LABELS[s] for s in rate.index])
for i in range(rate.shape[0]):
    for j in range(rate.shape[1]):
        ax.text(
            j,
            i,
            f"{rate.iloc[i, j]:.0%}\nn={size.iloc[i, j]}",
            ha="center",
            va="center",
        )
ax.set_xlabel("Age band (years)")
ax.set_title("Survival rate by age band and smoking status")
fig.colorbar(im, ax=ax, label="Proportion alive")
plt.tight_layout()
plt.show()

### 7. Is gender the explanation instead? (grouped bar)

In [ ]:
by_gender = df.groupby(["gender", "smoker"])["is_alive"].mean().unstack() * 100
by_gender = by_gender.rename(columns=LABELS)

ax = by_gender.plot.bar(color=[COLORS["No"], COLORS["Yes"]], figsize=(7, 5), rot=0)
ax.set_xlabel("Gender")
ax.set_ylabel("Proportion alive (%)")
ax.set_title("Survival rate by gender and smoking status")
ax.set_ylim(0, 100)
ax.legend(title="")
ax.grid(axis="y", alpha=0.3)
plt.show()

## Conclusion
**Age explains the paradox; smoking is not protective.**

- Smokers in this sample are younger: mean age is about 45 vs. 49 for non-smokers, and only about 17% of smokers are over 60 compared with about 33% of non-smokers (plots 2 and 3).
- Survival falls steeply with age, from about 98% for 18-30 year olds to about 29% for those over 60 (plot 4). A group with more old people will therefore show more deaths, whatever they smoke.
- Once we compare people of the *same* age, the advantage disappears. In 6 of the 7 age bands smokers do the same or worse than non-smokers, and the one band where they do slightly better (26-35) differs by only about a percentage point (plots 5 and 6).
- Gender is not the explanation: smokers have a similar or higher survival rate than non-smokers within both genders, so splitting by gender does not remove the effect (plot 7). Age does.

**Caveats:** this is observational data, the oldest bands contain few smokers (so those rates are noisy), and we have no information on follow-up time or smoking intensity. The data are consistent with smoking being harmful once age is accounted for, but they cannot prove it on their own. This is a textbook case of **Simpson's paradox**: a trend in the combined data that weakens or reverses once a confounder (age) is controlled for.

## Does salary explain the result?
Salary is only recorded for some people, so everything in this section uses only the rows that have a salary. The questions are:
1. Are non-smokers poorer than smokers?
2. Does a lower income lead to a higher chance of death?
3. Does the smoker vs. non-smoker survival gap hold up within income groups?

In [ ]:
# Keep only the rows with a salary and split them into income quartiles
sal = df.dropna(subset=["salary"]).copy()
sal["salary_quartile"] = pd.qcut(
    sal["salary"], 4, labels=["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"]
)

print(f"Rows with salary: {len(sal)} of {len(df)} ({len(sal) / len(df):.0%})")
print("Share of each group with salary missing:")
print(df.groupby("smoker")["salary"].apply(lambda s: s.isna().mean()).round(3))

### 1. Are non-smokers poorer? Salary distribution by smoking status (violin plot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = ["No", "Yes"]
parts = ax.violinplot(
    [sal.loc[sal["smoker"] == s, "salary"] for s in groups], showmedians=True
)
for body, s in zip(parts["bodies"], groups):
    body.set_facecolor(COLORS[s])
    body.set_alpha(0.6)
ax.set_xticks([1, 2], [LABELS[s] for s in groups])
ax.set_ylabel("Salary")
ax.set_title("Salary distribution by smoking status (line = median)")
ax.grid(axis="y", alpha=0.3)
plt.show()

### 2. The same question as cumulative curves (line plot)
If one curve sits to the left of the other, that group earns less at every point of the distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for s in ["No", "Yes"]:
    values = np.sort(sal.loc[sal["smoker"] == s, "salary"].to_numpy())
    share = np.arange(1, len(values) + 1) / len(values)
    ax.step(values, share, where="post", color=COLORS[s], label=LABELS[s])
ax.set_xlabel("Salary")
ax.set_ylabel("Share of group earning up to this salary")
ax.set_title("Cumulative salary distribution by smoking status")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### 3. Does lower income mean higher chance of death? (bar chart and box plot)

In [ ]:
by_quartile = sal.groupby("salary_quartile", observed=True)["is_alive"].agg(
    ["mean", "count"]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(
    by_quartile.index.astype(str), by_quartile["mean"] * 100, color="tab:green"
)
for bar, rate, n in zip(bars, by_quartile["mean"], by_quartile["count"]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{rate:.0%}\n(n={n})",
        ha="center",
    )
axes[0].set_ylim(0, 100)
axes[0].set_xlabel("Salary quartile")
axes[0].set_ylabel("Proportion alive (%)")
axes[0].set_title("Survival rate by salary quartile")
axes[0].grid(axis="y", alpha=0.3)

outcomes = ["Alive", "Dead"]
bp = axes[1].boxplot(
    [sal.loc[sal["outcome"] == o, "salary"] for o in outcomes],
    tick_labels=outcomes,
    patch_artist=True,
    showmeans=True,
)
for patch, color in zip(bp["boxes"], ["tab:green", "tab:red"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_xlabel("Outcome")
axes[1].set_ylabel("Salary")
axes[1].set_title("Salary by outcome (triangle = mean)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### 4. Is salary linked to age? Salary vs. age, coloured by outcome (scatter plot)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for outcome, color in [("Alive", "tab:green"), ("Dead", "tab:red")]:
    subset = sal[sal["outcome"] == outcome]
    ax.scatter(
        subset["age"],
        subset["salary"],
        alpha=0.5,
        color=color,
        label=outcome,
        edgecolor="none",
    )
ax.set_xlabel("Age (years)")
ax.set_ylabel("Salary")
ax.set_title("Salary vs. age, coloured by outcome")
ax.legend(title="Outcome")
ax.grid(alpha=0.3)
plt.show()

### 5. Does the smoker vs. non-smoker gap survive within income groups? (grouped bar)

In [ ]:
within_income = sal.groupby(["salary_quartile", "smoker"], observed=True)[
    "is_alive"
].agg(["mean", "count"])
rate = within_income["mean"].unstack() * 100
count = within_income["count"].unstack()

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(rate.index))
width = 0.38
for offset, s in zip([-width / 2, width / 2], ["No", "Yes"]):
    bars = ax.bar(x + offset, rate[s], width, color=COLORS[s], label=LABELS[s])
    for bar, n in zip(bars, count[s]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"n={n}",
            ha="center",
            fontsize=9,
        )
ax.set_xticks(x, rate.index.astype(str))
ax.set_ylim(0, 100)
ax.set_xlabel("Salary quartile")
ax.set_ylabel("Proportion alive (%)")
ax.set_title("Survival rate by salary quartile and smoking status")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()

## Salary conclusion
**Salary does not explain the result.**

- **Non-smokers are not noticeably poorer.** Among people with a salary, the median is about 47k for non-smokers and 50k for smokers, and the two distributions overlap almost completely (plots 1 and 2). If anything smokers earn slightly more, so lower income cannot be what makes non-smokers look worse off.
- **Lower income does not lead to a higher chance of death here.** Survival is about 70-75% in every salary quartile with no trend, and the average salary of people who died is almost the same as for those who survived (plot 3).
- **Salary is unrelated to age** in this data (plot 4), so it cannot be hiding the age effect either.
- **Splitting by income does not give a consistent story.** The smoker/non-smoker gap points in different directions in different quartiles (plot 5), but each bar rests on only about 60-90 people, so these differences are mostly noise. The smokers' survival advantage remains in the pooled data because they are younger, as shown in the age analysis above.

**Caveats:** over half of the rows (about 53%) have no salary, so these results describe a subset of the sample. The share missing is similar for smokers and non-smokers (roughly 54% vs. 52%), which suggests the missing data is not strongly biased, but we cannot be certain. Age remains the main explanation for the paradox; salary adds nothing to it.